# Clase 18 — DataFrames: Joins y Spark SQL

Notebook con **todos los ejercicios y casos de uso** de las dos sesiones de la Clase 18:

- **Sesión 1** — Joins y Uniones · Práctica *LogiData S.A.* + Caso de Estudio *MediRed S.A.*
- **Sesión 2** — Spark SQL · Práctica *FreshMart* + Caso de Estudio *EduTrack Academy*

Entorno: Apache Spark 4.1.1 + Scala 2.13 (kernel Almond) en modo `local[*]`.


---

# 🟢 Sesión 1 — DataFrames: Joins y Uniones

## 0. Inicialización de Spark


In [1]:
import $ivy.`org.apache.spark::spark-sql:4.1.1`
import org.apache.log4j.{Level, Logger}
Logger.getLogger("org").setLevel(Level.ERROR)
Logger.getLogger("akka").setLevel(Level.ERROR)

import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions._

val spark = SparkSession.builder()
  .appName("Clase18-Sesion1-Joins")
  .master("local[*]")
  .config("spark.sql.shuffle.partitions", "4")
  .config("spark.sql.crossJoin.enabled",  "true")
  .config("spark.ui.showConsoleProgress", "false")
  .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
import spark.implicits._

println(s"Spark ${spark.version} — Scala ${scala.util.Properties.versionString} ✅")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/03 23:16:19 INFO SparkContext: Running Spark version 4.1.1
26/05/03 23:16:19 INFO SparkContext: OS info Windows 11, 10.0, amd64
26/05/03 23:16:19 INFO SparkContext: Java version 17.0.18+8
26/05/03 23:16:19 INFO ResourceUtils: ==============================================================
26/05/03 23:16:19 INFO ResourceUtils: No custom resources configured for spark.driver.
26/05/03 23:16:19 INFO ResourceUtils: ==============================================================
26/05/03 23:16:19 INFO SparkContext: Submitted application: Clase18-Sesion1-Joins
26/05/03 23:16:19 INFO SecurityManager: Changing view acls to: gre
26/05/03 23:16:19 INFO SecurityManager: Changing modify acls to: gre
26/05/03 23:16:19 INFO SecurityManager: Changing view acls groups to: gre
26/05/03 23:16:19 INFO SecurityManager: Changing modify acls groups to: gre
26/05/03 23:16:19 INFO SecurityManager: SecurityManager: authenticat

Spark 4.1.1 — Scala version 2.13.17 ✅


import $ivy.$
import org.apache.log4j.{Level, Logger}
import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions._
spark: SparkSession = org.apache.spark.sql.classic.SparkSession@30cc0aca
import spark.implicits._

---

## 💻 Práctica — *LogiData S.A.*

Trabajamos con tres fuentes de datos: `clientes`, `pedidos` y `repartidores`.


### 🔹 Celda 2 — Crear los DataFrames de LogiData

In [2]:
// === CLIENTES ===
val clientes = Seq(
  (1, "Ana García",    "Madrid"),
  (2, "Borja Ruiz",    "Barcelona"),
  (3, "Carmen López",  "Sevilla"),
  (4, "David Mora",    "Valencia"),
  (5, "Elena Pardo",   "Bilbao")
).toDF("clienteId", "nombre", "ciudad")

// === PEDIDOS ===  (clienteId 6 no existe en el maestro → huérfano)
val pedidos = Seq(
  ("P001", 1, "2024-01-10", 250.0, "R01"),
  ("P002", 1, "2024-01-15", 180.0, "R02"),
  ("P003", 2, "2024-01-20", 420.0, "R01"),
  ("P004", 3, "2024-02-01",  95.0, "R03"),
  ("P005", 3, "2024-02-14",  60.0, "R02"),
  ("P006", 6, "2024-02-20", 310.0, "R03")
).toDF("pedidoId", "clienteId", "fecha", "importe", "repartidorId")

// === REPARTIDORES ===
val repartidores = Seq(
  ("R01", "Miguel Sanz",   "Zona Norte"),
  ("R02", "Laura Vega",    "Zona Sur"),
  ("R03", "Pablo Fuentes", "Zona Este")
).toDF("repartidorId", "nombreRep", "zona")

println(s"clientes: ${clientes.count()} | pedidos: ${pedidos.count()} | repartidores: ${repartidores.count()}")

clientes: 5 | pedidos: 6 | repartidores: 3


clientes: org.apache.spark.sql.package.DataFrame = [clienteId: int, nombre: string ... 1 more field]
pedidos: org.apache.spark.sql.package.DataFrame = [pedidoId: string, clienteId: int ... 3 more fields]
repartidores: org.apache.spark.sql.package.DataFrame = [repartidorId: string, nombreRep: string ... 1 more field]

### 🔹 Celda 3 — INNER JOIN: pedidos con cliente conocido

In [3]:
val pedidosRen = pedidos.withColumnRenamed("clienteId", "pedido_clienteId")

val resultado_inner = clientes.join(
  pedidosRen,
  col("clienteId") === col("pedido_clienteId"),
  "inner"
).select(
  col("pedidoId"),
  col("nombre").alias("cliente"),
  col("ciudad"),
  col("fecha"),
  col("importe")
).orderBy("pedidoId")

println("=== INNER JOIN ===")
resultado_inner.show()
println(s"Total filas: ${resultado_inner.count()}")

=== INNER JOIN ===
+--------+------------+---------+----------+-------+
|pedidoId|     cliente|   ciudad|     fecha|importe|
+--------+------------+---------+----------+-------+
|    P001|  Ana García|   Madrid|2024-01-10|  250.0|
|    P002|  Ana García|   Madrid|2024-01-15|  180.0|
|    P003|  Borja Ruiz|Barcelona|2024-01-20|  420.0|
|    P004|Carmen López|  Sevilla|2024-02-01|   95.0|
|    P005|Carmen López|  Sevilla|2024-02-14|   60.0|
+--------+------------+---------+----------+-------+

Total filas: 5


pedidosRen: org.apache.spark.sql.package.DataFrame = [pedidoId: string, pedido_clienteId: int ... 3 more fields]
resultado_inner: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [pedidoId: string, cliente: string ... 3 more fields]

### 🔹 Celda 4 — LEFT JOIN vs RIGHT JOIN

In [4]:
// LEFT: todos los clientes (con o sin pedidos)
val resultado_left = clientes.join(
  pedidosRen,
  col("clienteId") === col("pedido_clienteId"),
  "left"
).select(
  col("clienteId"),
  col("nombre"),
  col("pedidoId"),
  col("importe")
).orderBy("clienteId", "pedidoId")

println("=== LEFT JOIN ===")
resultado_left.show()

// RIGHT: todos los pedidos (incluido el huérfano)
val resultado_right = clientes.join(
  pedidosRen,
  col("clienteId") === col("pedido_clienteId"),
  "right"
).select(
  col("pedido_clienteId").alias("clienteId"),
  col("nombre"),
  col("pedidoId"),
  col("importe")
).orderBy("pedidoId")

println("=== RIGHT JOIN ===")
resultado_right.show()

=== LEFT JOIN ===
+---------+------------+--------+-------+
|clienteId|      nombre|pedidoId|importe|
+---------+------------+--------+-------+
|        1|  Ana García|    P001|  250.0|
|        1|  Ana García|    P002|  180.0|
|        2|  Borja Ruiz|    P003|  420.0|
|        3|Carmen López|    P004|   95.0|
|        3|Carmen López|    P005|   60.0|
|        4|  David Mora|    NULL|   NULL|
|        5| Elena Pardo|    NULL|   NULL|
+---------+------------+--------+-------+

=== RIGHT JOIN ===
+---------+------------+--------+-------+
|clienteId|      nombre|pedidoId|importe|
+---------+------------+--------+-------+
|        1|  Ana García|    P001|  250.0|
|        1|  Ana García|    P002|  180.0|
|        2|  Borja Ruiz|    P003|  420.0|
|        3|Carmen López|    P004|   95.0|
|        3|Carmen López|    P005|   60.0|
|        6|        NULL|    P006|  310.0|
+---------+------------+--------+-------+



resultado_left: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [clienteId: int, nombre: string ... 2 more fields]
resultado_right: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [clienteId: int, nombre: string ... 2 more fields]

### 🔹 Celda 5 — SEMI y ANTI JOIN

In [5]:
// SEMI: clientes que SÍ tienen pedidos
val clientesConPedidos = clientes.join(
  pedidosRen,
  col("clienteId") === col("pedido_clienteId"),
  "semi"
)
println("=== LEFT SEMI ===")
clientesConPedidos.show()

// ANTI: clientes sin ningún pedido
val clientesSinPedidos = clientes.join(
  pedidosRen,
  col("clienteId") === col("pedido_clienteId"),
  "anti"
)
println("=== LEFT ANTI ===")
clientesSinPedidos.show()

=== LEFT SEMI ===
+---------+------------+---------+
|clienteId|      nombre|   ciudad|
+---------+------------+---------+
|        1|  Ana García|   Madrid|
|        2|  Borja Ruiz|Barcelona|
|        3|Carmen López|  Sevilla|
+---------+------------+---------+

=== LEFT ANTI ===
+---------+-----------+--------+
|clienteId|     nombre|  ciudad|
+---------+-----------+--------+
|        4| David Mora|Valencia|
|        5|Elena Pardo|  Bilbao|
+---------+-----------+--------+



clientesConPedidos: org.apache.spark.sql.package.DataFrame = [clienteId: int, nombre: string ... 1 more field]
clientesSinPedidos: org.apache.spark.sql.package.DataFrame = [clienteId: int, nombre: string ... 1 more field]

### 🔹 Celda 6 — JOIN de tres tablas

In [6]:
val pedidos3 = pedidos
  .withColumnRenamed("clienteId",    "pedido_clienteId")
  .withColumnRenamed("repartidorId", "pedido_repartidorId")

val resultado_triple = pedidos3
  .join(clientes,     col("pedido_clienteId")    === col("clienteId"),    "inner")
  .join(repartidores, col("pedido_repartidorId") === col("repartidorId"), "inner")
  .select(
    col("pedidoId"),
    col("nombre").alias("cliente"),
    col("ciudad"),
    col("fecha"),
    col("importe"),
    col("nombreRep").alias("repartidor"),
    col("zona")
  )
  .orderBy("pedidoId")

println("=== JOIN TRIPLE ===")
resultado_triple.show(truncate = false)

=== JOIN TRIPLE ===
+--------+------------+---------+----------+-------+-------------+----------+
|pedidoId|cliente     |ciudad   |fecha     |importe|repartidor   |zona      |
+--------+------------+---------+----------+-------+-------------+----------+
|P001    |Ana García  |Madrid   |2024-01-10|250.0  |Miguel Sanz  |Zona Norte|
|P002    |Ana García  |Madrid   |2024-01-15|180.0  |Laura Vega   |Zona Sur  |
|P003    |Borja Ruiz  |Barcelona|2024-01-20|420.0  |Miguel Sanz  |Zona Norte|
|P004    |Carmen López|Sevilla  |2024-02-01|95.0   |Pablo Fuentes|Zona Este |
|P005    |Carmen López|Sevilla  |2024-02-14|60.0   |Laura Vega   |Zona Sur  |
+--------+------------+---------+----------+-------+-------------+----------+



pedidos3: org.apache.spark.sql.package.DataFrame = [pedidoId: string, pedido_clienteId: int ... 3 more fields]
resultado_triple: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [pedidoId: string, cliente: string ... 5 more fields]

### 🔹 Celda 7 — Broadcast JOIN y comparación de planes

In [7]:
val pedidos4 = pedidos
  .withColumnRenamed("repartidorId", "pedido_repartidorId")

val sinBroadcast = pedidos4.join(
  repartidores,
  col("pedido_repartidorId") === col("repartidorId"),
  "inner"
)
println("=== Plan SIN broadcast ===")
sinBroadcast.explain()

val conBroadcast = pedidos4.join(
  broadcast(repartidores),
  col("pedido_repartidorId") === col("repartidorId"),
  "inner"
)
println("\n=== Plan CON broadcast ===")
conBroadcast.explain()

println(s"\nFilas sin broadcast: ${sinBroadcast.count()}")
println(s"Filas con broadcast: ${conBroadcast.count()}")

=== Plan SIN broadcast ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- BroadcastHashJoin [pedido_repartidorId#181], [repartidorId#55], Inner, BuildRight, false
   :- LocalTableScan [pedidoId#37, clienteId#38, fecha#39, importe#40, pedido_repartidorId#181]
   +- BroadcastExchange HashedRelationBroadcastMode(List(input[0, string, true]),false), [plan_id=517]
      +- LocalTableScan [repartidorId#55, nombreRep#56, zona#57]



=== Plan CON broadcast ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- BroadcastHashJoin [pedido_repartidorId#181], [repartidorId#55], Inner, BuildRight, false
   :- LocalTableScan [pedidoId#37, clienteId#38, fecha#39, importe#40, pedido_repartidorId#181]
   +- BroadcastExchange HashedRelationBroadcastMode(List(input[0, string, true]),false), [plan_id=531]
      +- LocalTableScan [repartidorId#55, nombreRep#56, zona#57]



Filas sin broadcast: 6
Filas con broadcast: 6


pedidos4: org.apache.spark.sql.package.DataFrame = [pedidoId: string, clienteId: int ... 3 more fields]
sinBroadcast: org.apache.spark.sql.package.DataFrame = [pedidoId: string, clienteId: int ... 6 more fields]
conBroadcast: org.apache.spark.sql.package.DataFrame = [pedidoId: string, clienteId: int ... 6 more fields]

### 🔹 Celda 8 — `union`, `distinct` y `except`

In [8]:
val ciudadesEnero = Seq(
  ("Madrid",    "Norte"),
  ("Barcelona", "Este"),
  ("Sevilla",   "Sur")
).toDF("ciudad", "zona")

val ciudadesFebrero = Seq(
  ("Madrid",   "Norte"),
  ("Valencia", "Este"),
  ("Bilbao",   "Norte")
).toDF("ciudad", "zona")

val todasConDup = ciudadesEnero.union(ciudadesFebrero)
println(s"=== union (con duplicados): ${todasConDup.count()} filas ===")
todasConDup.show()

val todasSinDup = ciudadesEnero.union(ciudadesFebrero).distinct()
println(s"=== union.distinct: ${todasSinDup.count()} filas ===")
todasSinDup.show()

val soloEnero = ciudadesEnero.except(ciudadesFebrero)
println("=== except: solo en enero ===")
soloEnero.show()

=== union (con duplicados): 6 filas ===
+---------+-----+
|   ciudad| zona|
+---------+-----+
|   Madrid|Norte|
|Barcelona| Este|
|  Sevilla|  Sur|
|   Madrid|Norte|
| Valencia| Este|
|   Bilbao|Norte|
+---------+-----+

=== union.distinct: 5 filas ===
+---------+-----+
|   ciudad| zona|
+---------+-----+
|   Madrid|Norte|
|Barcelona| Este|
|  Sevilla|  Sur|
| Valencia| Este|
|   Bilbao|Norte|
+---------+-----+

=== except: solo en enero ===
+---------+----+
|   ciudad|zona|
+---------+----+
|Barcelona|Este|
|  Sevilla| Sur|
+---------+----+



ciudadesEnero: org.apache.spark.sql.package.DataFrame = [ciudad: string, zona: string]
ciudadesFebrero: org.apache.spark.sql.package.DataFrame = [ciudad: string, zona: string]
todasConDup: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [ciudad: string, zona: string]
todasSinDup: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [ciudad: string, zona: string]
soloEnero: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [ciudad: string, zona: string]

### 🔹 Celda 9 — Verificación final LogiData

In [9]:
println("=" * 52)
println("RESUMEN — Sesión 1 | LogiData S.A.")
println("=" * 52)

val checks = Seq(
  ("INNER JOIN — pedidos con cliente válido",     resultado_inner.count()    == 5),
  ("LEFT ANTI  — clientes sin pedidos",           clientesSinPedidos.count() == 2),
  ("JOIN TRIPLE — pedidos+clientes+repartidores", resultado_triple.count()   == 5),
  ("UNION.DISTINCT — sin duplicados",             todasSinDup.count()        == 5),
  ("EXCEPT — solo ciudades de enero",             soloEnero.count()          == 2)
)

checks.foreach { case (desc, ok) =>
  println(s"${if (ok) "✅ CORRECTO" else "❌ REVISAR"} — $desc")
}

RESUMEN — Sesión 1 | LogiData S.A.
✅ CORRECTO — INNER JOIN — pedidos con cliente válido
✅ CORRECTO — LEFT ANTI  — clientes sin pedidos
✅ CORRECTO — JOIN TRIPLE — pedidos+clientes+repartidores
✅ CORRECTO — UNION.DISTINCT — sin duplicados
✅ CORRECTO — EXCEPT — solo ciudades de enero


checks: Seq[(String, Boolean)] = List(
  ("INNER JOIN — pedidos con cliente válido", true),
  ("LEFT ANTI  — clientes sin pedidos", true),
  ("JOIN TRIPLE — pedidos+clientes+repartidores", true),
  ("UNION.DISTINCT — sin duplicados", true),
  ("EXCEPT — solo ciudades de enero", true)
)

---

## 🏥 Caso de Estudio — *MediRed S.A.*

Red nacional de farmacias con cuatro fuentes: `farmacias`, `ventas`, `productos`, `inspeccionesQ1`/`inspeccionesQ2`. El sistema de ventas a veces registra `farmaciaId` huérfanos (proveedor externo).


### Datos de MediRed S.A.

In [10]:
// === FARMACIAS ===
val farmacias = Seq(
  (1, "Farmacia Central",  "Madrid",    "Madrid",     "Laura Vega"),
  (2, "Farmacia del Sol",  "Madrid",    "Madrid",     "Carlos Ruiz"),
  (3, "Farmacia Diagonal", "Barcelona", "Cataluña",   "Ana Puig"),
  (4, "Farmacia Rambla",   "Barcelona", "Cataluña",   "Marta Font"),
  (5, "Farmacia Norte",    "Bilbao",    "País Vasco", "Jon Etxea"),
  (6, "Farmacia Ría",      "Bilbao",    "País Vasco", "Amaia Goñi"),
  (7, "Farmacia Gran Vía", "Sevilla",   "Andalucía",  "Pedro Mora"),
  (8, "Farmacia Triana",   "Sevilla",   "Andalucía",  "Rosa Leal")
).toDF("farmaciaId", "nombre", "ciudad", "comunidad", "titular")

// === VENTAS === (V012 con farmaciaId 99 → huérfano)
val ventas = Seq(
  ("V001",  1, "P-AMOX", 3, 18.50, "2024-01-05"),
  ("V002",  1, "P-IBUP",10, 42.00, "2024-01-07"),
  ("V003",  2, "P-VITA", 5, 25.00, "2024-01-08"),
  ("V004",  3, "P-AMOX", 2, 12.50, "2024-01-10"),
  ("V005",  3, "P-PARA", 8, 16.00, "2024-01-12"),
  ("V006",  4, "P-IBUP", 6, 24.00, "2024-01-15"),
  ("V007",  5, "P-VITA",12, 60.00, "2024-01-18"),
  ("V008",  6, "P-PARA", 4,  8.00, "2024-01-20"),
  ("V009",  7, "P-AMOX", 7, 43.75, "2024-02-01"),
  ("V010",  7, "P-IBUP", 3, 12.00, "2024-02-03"),
  ("V011",  8, "P-VITA", 9, 45.00, "2024-02-05"),
  ("V012", 99, "P-PARA", 2,  4.00, "2024-02-10")
).toDF("ventaId", "farmaciaId", "productoId", "cantidad", "importe", "fecha")

// === PRODUCTOS === (tabla pequeña)
val productos = Seq(
  ("P-AMOX", "Amoxicilina 500mg",  "Antibiótico", true),
  ("P-IBUP", "Ibuprofeno 600mg",   "Analgésico",  false),
  ("P-VITA", "Vitamina D 1000 UI", "Vitamina",    false),
  ("P-PARA", "Paracetamol 1g",     "Analgésico",  false)
).toDF("productoId", "nombreProducto", "categoria", "requiereReceta")

// === INSPECCIONES Q1 ===
val inspeccionesQ1 = Seq(
  (1, "2024-01-15", "Apto"),
  (2, "2024-01-22", "Apto"),
  (3, "2024-02-05", "No Apto"),
  (5, "2024-02-18", "Apto"),
  (7, "2024-03-10", "Apto")
).toDF("farmaciaId", "fecha", "resultado")

// === INSPECCIONES Q2 ===
val inspeccionesQ2 = Seq(
  (3, "2024-04-08", "Apto"),
  (4, "2024-04-20", "Apto"),
  (6, "2024-05-12", "No Apto"),
  (7, "2024-05-25", "Apto"),
  (8, "2024-06-03", "Apto")
).toDF("farmaciaId", "fecha", "resultado")

println("Datos cargados:")
println(s"  farmacias:      ${farmacias.count()}")
println(s"  ventas:         ${ventas.count()}")
println(s"  productos:      ${productos.count()}")
println(s"  inspeccionesQ1: ${inspeccionesQ1.count()}")
println(s"  inspeccionesQ2: ${inspeccionesQ2.count()}")

Datos cargados:
  farmacias:      8
  ventas:         12
  productos:      4
  inspeccionesQ1: 5
  inspeccionesQ2: 5


farmacias: org.apache.spark.sql.package.DataFrame = [farmaciaId: int, nombre: string ... 3 more fields]
ventas: org.apache.spark.sql.package.DataFrame = [ventaId: string, farmaciaId: int ... 4 more fields]
productos: org.apache.spark.sql.package.DataFrame = [productoId: string, nombreProducto: string ... 2 more fields]
inspeccionesQ1: org.apache.spark.sql.package.DataFrame = [farmaciaId: int, fecha: string ... 1 more field]
inspeccionesQ2: org.apache.spark.sql.package.DataFrame = [farmaciaId: int, fecha: string ... 1 more field]

### 🎯 Misión 1 — Informe de ventas con nombre de producto

Cruzar `ventas` con `productos` (broadcast porque `productos` es pequeño).


In [11]:
val productosRen = productos.withColumnRenamed("productoId", "prod_productoId")

val informeVentasProducto = ventas
  .join(broadcast(productosRen),
        ventas("productoId") === productosRen("prod_productoId"),
        "inner")
  .select(
    col("ventaId"),
    col("nombreProducto"),
    col("categoria"),
    col("requiereReceta"),
    col("cantidad"),
    col("importe")
  )
  .orderBy(col("importe").desc)

println("=== Misión 1 ===")
informeVentasProducto.show(truncate = false)
println(s"Total filas: ${informeVentasProducto.count()}")

=== Misión 1 ===
+-------+------------------+-----------+--------------+--------+-------+
|ventaId|nombreProducto    |categoria  |requiereReceta|cantidad|importe|
+-------+------------------+-----------+--------------+--------+-------+
|V007   |Vitamina D 1000 UI|Vitamina   |false         |12      |60.0   |
|V011   |Vitamina D 1000 UI|Vitamina   |false         |9       |45.0   |
|V009   |Amoxicilina 500mg |Antibiótico|true          |7       |43.75  |
|V002   |Ibuprofeno 600mg  |Analgésico |false         |10      |42.0   |
|V003   |Vitamina D 1000 UI|Vitamina   |false         |5       |25.0   |
|V006   |Ibuprofeno 600mg  |Analgésico |false         |6       |24.0   |
|V001   |Amoxicilina 500mg |Antibiótico|true          |3       |18.5   |
|V005   |Paracetamol 1g    |Analgésico |false         |8       |16.0   |
|V004   |Amoxicilina 500mg |Antibiótico|true          |2       |12.5   |
|V010   |Ibuprofeno 600mg  |Analgésico |false         |3       |12.0   |
|V008   |Paracetamol 1g    |Analgé

productosRen: org.apache.spark.sql.package.DataFrame = [prod_productoId: string, nombreProducto: string ... 2 more fields]
informeVentasProducto: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [ventaId: string, nombreProducto: string ... 4 more fields]

### 🎯 Misión 2 — Farmacias sin ventas registradas (LEFT ANTI)

In [12]:
val ventasRen = ventas.withColumnRenamed("farmaciaId", "venta_farmaciaId")

val farmaciasSinVentas = farmacias
  .join(ventasRen,
        farmacias("farmaciaId") === ventasRen("venta_farmaciaId"),
        "anti")
  .select("farmaciaId", "nombre", "ciudad")

println("=== Misión 2 ===")
farmaciasSinVentas.show()
println(s"Farmacias sin ventas: ${farmaciasSinVentas.count()}")

=== Misión 2 ===
+----------+------+------+
|farmaciaId|nombre|ciudad|
+----------+------+------+
+----------+------+------+

Farmacias sin ventas: 0


ventasRen: org.apache.spark.sql.package.DataFrame = [ventaId: string, venta_farmaciaId: int ... 4 more fields]
farmaciasSinVentas: org.apache.spark.sql.package.DataFrame = [farmaciaId: int, nombre: string ... 1 more field]

### 🎯 Misión 3 — Detectar ventas huérfanas (LEFT ANTI invertido)

In [13]:
val farmaciasRen = farmacias.withColumnRenamed("farmaciaId", "f_farmaciaId")

val ventasHuerfanas = ventas
  .join(farmaciasRen,
        ventas("farmaciaId") === farmaciasRen("f_farmaciaId"),
        "anti")
  .select("ventaId", "farmaciaId", "productoId", "importe")

println("=== Misión 3 ===")
ventasHuerfanas.show()
println(s"Ventas huérfanas: ${ventasHuerfanas.count()}")

=== Misión 3 ===
+-------+----------+----------+-------+
|ventaId|farmaciaId|productoId|importe|
+-------+----------+----------+-------+
|   V012|        99|    P-PARA|    4.0|
+-------+----------+----------+-------+

Ventas huérfanas: 1


farmaciasRen: org.apache.spark.sql.package.DataFrame = [f_farmaciaId: int, nombre: string ... 3 more fields]
ventasHuerfanas: org.apache.spark.sql.package.DataFrame = [ventaId: string, farmaciaId: int ... 2 more fields]

### 🎯 Misión 4 — Informe completo: ventas + farmacia + producto

In [14]:
val ventasM4 = ventas
  .withColumnRenamed("farmaciaId", "v_farmaciaId")
  .withColumnRenamed("productoId", "v_productoId")

val informeCompleto = ventasM4
  .join(farmacias,        col("v_farmaciaId") === col("farmaciaId"), "inner")
  .join(broadcast(productos), col("v_productoId") === col("productoId"), "inner")
  .select(
    col("ventaId"),
    col("nombre"),
    col("comunidad"),
    col("nombreProducto"),
    col("categoria"),
    col("cantidad"),
    col("importe"),
    col("fecha")
  )
  .orderBy(col("comunidad"), col("importe").desc)

println("=== Misión 4 ===")
informeCompleto.show(truncate = false)
println(s"Total filas: ${informeCompleto.count()}")

=== Misión 4 ===
+-------+-----------------+----------+------------------+-----------+--------+-------+----------+
|ventaId|nombre           |comunidad |nombreProducto    |categoria  |cantidad|importe|fecha     |
+-------+-----------------+----------+------------------+-----------+--------+-------+----------+
|V011   |Farmacia Triana  |Andalucía |Vitamina D 1000 UI|Vitamina   |9       |45.0   |2024-02-05|
|V009   |Farmacia Gran Vía|Andalucía |Amoxicilina 500mg |Antibiótico|7       |43.75  |2024-02-01|
|V010   |Farmacia Gran Vía|Andalucía |Ibuprofeno 600mg  |Analgésico |3       |12.0   |2024-02-03|
|V006   |Farmacia Rambla  |Cataluña  |Ibuprofeno 600mg  |Analgésico |6       |24.0   |2024-01-15|
|V005   |Farmacia Diagonal|Cataluña  |Paracetamol 1g    |Analgésico |8       |16.0   |2024-01-12|
|V004   |Farmacia Diagonal|Cataluña  |Amoxicilina 500mg |Antibiótico|2       |12.5   |2024-01-10|
|V002   |Farmacia Central |Madrid    |Ibuprofeno 600mg  |Analgésico |10      |42.0   |2024-01-07|
|V0

ventasM4: org.apache.spark.sql.package.DataFrame = [ventaId: string, v_farmaciaId: int ... 4 more fields]
informeCompleto: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [ventaId: string, nombre: string ... 6 more fields]

### 🎯 Misión 5 — Farmacias inspeccionadas en ambos trimestres (`intersect`)

In [15]:
val q1Ids = inspeccionesQ1.select("farmaciaId")
val q2Ids = inspeccionesQ2.select("farmaciaId")

val farmaciasAmbosTrimsIds = q1Ids.intersect(q2Ids)

val farmaciasAmbosTrims = farmaciasAmbosTrimsIds
  .join(farmacias, Seq("farmaciaId"), "inner")
  .select("farmaciaId", "nombre", "ciudad")
  .orderBy("farmaciaId")

println("=== Misión 5 ===")
farmaciasAmbosTrims.show()
println(s"Total: ${farmaciasAmbosTrims.count()}")

=== Misión 5 ===
+----------+-----------------+---------+
|farmaciaId|           nombre|   ciudad|
+----------+-----------------+---------+
|         3|Farmacia Diagonal|Barcelona|
|         7|Farmacia Gran Vía|  Sevilla|
+----------+-----------------+---------+

Total: 2


q1Ids: org.apache.spark.sql.package.DataFrame = [farmaciaId: int]
q2Ids: org.apache.spark.sql.package.DataFrame = [farmaciaId: int]
farmaciasAmbosTrimsIds: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [farmaciaId: int]
farmaciasAmbosTrims: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [farmaciaId: int, nombre: string ... 1 more field]

### 🎯 Misión 6 — Historial completo del año (`union` + `distinct`)

In [16]:
val historialInspecciones = inspeccionesQ1
  .union(inspeccionesQ2)
  .distinct()
  .orderBy("fecha")

println("=== Misión 6 ===")
historialInspecciones.show()
println(s"Total filas únicas: ${historialInspecciones.count()}")

=== Misión 6 ===
+----------+----------+---------+
|farmaciaId|     fecha|resultado|
+----------+----------+---------+
|         1|2024-01-15|     Apto|
|         2|2024-01-22|     Apto|
|         3|2024-02-05|  No Apto|
|         5|2024-02-18|     Apto|
|         7|2024-03-10|     Apto|
|         3|2024-04-08|     Apto|
|         4|2024-04-20|     Apto|
|         6|2024-05-12|  No Apto|
|         7|2024-05-25|     Apto|
|         8|2024-06-03|     Apto|
+----------+----------+---------+

Total filas únicas: 10


historialInspecciones: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [farmaciaId: int, fecha: string ... 1 more field]

### 🎯 Misión 7 — Farmacias inspeccionadas solo en Q1 (`except`)

In [17]:
val soloQ1Ids = inspeccionesQ1.select("farmaciaId")
  .except(inspeccionesQ2.select("farmaciaId"))

val soloQ1 = soloQ1Ids
  .join(farmacias, Seq("farmaciaId"), "inner")
  .select("farmaciaId", "nombre", "titular")
  .orderBy("farmaciaId")

println("=== Misión 7 ===")
soloQ1.show()
println(s"Total: ${soloQ1.count()}")

=== Misión 7 ===
+----------+----------------+-----------+
|farmaciaId|          nombre|    titular|
+----------+----------------+-----------+
|         1|Farmacia Central| Laura Vega|
|         2|Farmacia del Sol|Carlos Ruiz|
|         5|  Farmacia Norte|  Jon Etxea|
+----------+----------------+-----------+

Total: 3


soloQ1Ids: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [farmaciaId: int]
soloQ1: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [farmaciaId: int, nombre: string ... 1 more field]

### 🎯 Misión 8 — Plan de auditoría cruzado (CROSS JOIN)

In [18]:
val farmaciasMin   = farmacias.select("farmaciaId", "nombre")
val categoriasUnicas = productos.select("categoria").distinct()

val matrizAuditoria = farmaciasMin
  .join(categoriasUnicas, lit(true), "cross")
  .orderBy("farmaciaId")

println("=== Misión 8 ===")
matrizAuditoria.show(50)
println(s"Total filas: ${matrizAuditoria.count()} (esperado 8 × 3 = 24)")

=== Misión 8 ===
+----------+-----------------+-----------+
|farmaciaId|           nombre|  categoria|
+----------+-----------------+-----------+
|         1| Farmacia Central|Antibiótico|
|         1| Farmacia Central| Analgésico|
|         1| Farmacia Central|   Vitamina|
|         2| Farmacia del Sol|Antibiótico|
|         2| Farmacia del Sol| Analgésico|
|         2| Farmacia del Sol|   Vitamina|
|         3|Farmacia Diagonal|Antibiótico|
|         3|Farmacia Diagonal| Analgésico|
|         3|Farmacia Diagonal|   Vitamina|
|         4|  Farmacia Rambla|Antibiótico|
|         4|  Farmacia Rambla| Analgésico|
|         4|  Farmacia Rambla|   Vitamina|
|         5|   Farmacia Norte|Antibiótico|
|         5|   Farmacia Norte| Analgésico|
|         5|   Farmacia Norte|   Vitamina|
|         6|     Farmacia Ría|Antibiótico|
|         6|     Farmacia Ría| Analgésico|
|         6|     Farmacia Ría|   Vitamina|
|         7|Farmacia Gran Vía|Antibiótico|
|         7|Farmacia Gran Vía| Analgé

farmaciasMin: org.apache.spark.sql.package.DataFrame = [farmaciaId: int, nombre: string]
categoriasUnicas: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [categoria: string]
matrizAuditoria: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [farmaciaId: int, nombre: string ... 1 more field]

### 🎯 Misión 9 — Análisis del plan de ejecución

In [19]:
// Sin broadcast
val joinSinBroadcast = ventas.join(
  productos,
  ventas("productoId") === productos("productoId"),
  "inner"
)
println("=== SIN broadcast ===")
joinSinBroadcast.explain()

// Con broadcast
val joinConBroadcast = ventas.join(
  broadcast(productos),
  ventas("productoId") === productos("productoId"),
  "inner"
)
println("\n=== CON broadcast ===")
joinConBroadcast.explain()

=== SIN broadcast ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- BroadcastHashJoin [productoId#355], [productoId#376], Inner, BuildRight, false
   :- LocalTableScan [ventaId#353, farmaciaId#354, productoId#355, cantidad#356, importe#357, fecha#358]
   +- BroadcastExchange HashedRelationBroadcastMode(List(input[0, string, true]),false), [plan_id=3817]
      +- LocalTableScan [productoId#376, nombreProducto#377, categoria#378, requiereReceta#379]



=== CON broadcast ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- BroadcastHashJoin [productoId#355], [productoId#376], Inner, BuildRight, false
   :- LocalTableScan [ventaId#353, farmaciaId#354, productoId#355, cantidad#356, importe#357, fecha#358]
   +- BroadcastExchange HashedRelationBroadcastMode(List(input[0, string, true]),false), [plan_id=3831]
      +- LocalTableScan [productoId#376, nombreProducto#377, categoria#378, requiereReceta#379]




joinSinBroadcast: org.apache.spark.sql.package.DataFrame = [ventaId: string, farmaciaId: int ... 8 more fields]
joinConBroadcast: org.apache.spark.sql.package.DataFrame = [ventaId: string, farmaciaId: int ... 8 more fields]

#### 📝 Respuestas — Misión 9

1. **¿Qué tipo de join aparece sin broadcast?** En un cluster real con datasets grandes aparecería normalmente un `SortMergeJoin` (requiere shuffle por la clave del join). En modo `local[*]` con tablas tan pequeñas Spark suele aplicar `BroadcastHashJoin` automáticamente porque ambas caben bajo el umbral por defecto (`spark.sql.autoBroadcastJoinThreshold = 10MB`).
2. **¿Qué tipo de join aparece con broadcast?** `BroadcastHashJoin` — Spark envía la tabla `productos` completa a cada executor y hace una sonda hash en memoria contra `ventas`, evitando el shuffle.
3. **¿Por qué broadcast sobre `productos` y no sobre `ventas`?** Porque `productos` es una tabla pequeña (4 filas, KB) que cabe holgadamente en la memoria de cada nodo. `ventas` puede crecer a millones de filas; transmitirla por la red sería prohibitivo y eliminaría las ventajas de la paralelización.
4. **¿Qué columna de `ventas` es la adecuada para el join con `productos`?** `productoId` — es la clave foránea que referencia al catálogo (`productos.productoId`).


### ✅ Verificación final — MediRed

In [20]:
println("=" * 55)
println("VERIFICACIÓN FINAL — Caso de Estudio MediRed S.A.")
println("=" * 55)

val comprobaciones = Seq(
  ("Misión 3 — Ventas huérfanas detectadas",   ventasHuerfanas.count()    == 1),
  ("Misión 5 — Farmacias con doble inspección", farmaciasAmbosTrims.count() == 2),
  ("Misión 7 — Farmacias solo en Q1",           soloQ1.count()              == 3),
  ("Misión 8 — Matriz auditoría (8 × 3 = 24)",  matrizAuditoria.count()     == 24)
)

comprobaciones.foreach { case (desc, ok) =>
  println(s"${if (ok) "✅ CORRECTO" else "❌ REVISAR"} — $desc")
}

VERIFICACIÓN FINAL — Caso de Estudio MediRed S.A.
✅ CORRECTO — Misión 3 — Ventas huérfanas detectadas
✅ CORRECTO — Misión 5 — Farmacias con doble inspección
✅ CORRECTO — Misión 7 — Farmacias solo en Q1
✅ CORRECTO — Misión 8 — Matriz auditoría (8 × 3 = 24)


comprobaciones: Seq[(String, Boolean)] = List(
  ("Misión 3 — Ventas huérfanas detectadas", true),
  ("Misión 5 — Farmacias con doble inspección", true),
  ("Misión 7 — Farmacias solo en Q1", true),
  ("Misión 8 — Matriz auditoría (8 × 3 = 24)", true)
)

---

# 🟣 Sesión 2 — Spark SQL

Trabajaremos con vistas temporales (`createOrReplaceTempView`), subconsultas (`IN`, `EXISTS`, FROM, escalar) y CTEs.


## 💻 Práctica — *FreshMart*

Cadena de supermercados con tres regiones. Tres fuentes: `tiendas`, `productos` y `ventas`. Mismo `SparkSession` ya creado.


### 🔹 Celda 2 — Datos de FreshMart

In [21]:
// === TIENDAS ===
val tiendas = Seq(
  (1, "FreshMart Castellana",  "Madrid",    "Centro"),
  (2, "FreshMart Vallecas",    "Madrid",    "Centro"),
  (3, "FreshMart Diagonal",    "Barcelona", "Este"),
  (4, "FreshMart Gracia",      "Barcelona", "Este"),
  (5, "FreshMart Nervión",     "Bilbao",    "Norte"),
  (6, "FreshMart Casco Viejo", "Bilbao",    "Norte")
).toDF("tiendaId", "nombre", "ciudad", "region")

// === PRODUCTOS ===
val productosFM = Seq(
  ("P01", "Leche entera 1L",    "Lácteos",    0.89),
  ("P02", "Pan de molde",       "Panadería",  1.25),
  ("P03", "Yogur natural x4",   "Lácteos",    1.80),
  ("P04", "Zumo naranja 1L",    "Bebidas",    1.50),
  ("P05", "Agua mineral 1.5L",  "Bebidas",    0.45),
  ("P06", "Pollo entero",       "Carnicería", 6.90),
  ("P07", "Filete de ternera",  "Carnicería", 9.50),
  ("P08", "Manzanas bolsa 1kg", "Frutas",     2.20),
  ("P09", "Tomates rama 500g",  "Verduras",   1.95),
  ("P10", "Aceite oliva 1L",    "Aceites",    5.80)
).toDF("productoId", "nombreProducto", "categoria", "precioUnitario")

// === VENTAS (enero-marzo 2024) ===
val ventasFM = Seq(
  ("V001", 1, "P01", 120, "2024-01-05"), ("V002", 1, "P06",  18, "2024-01-05"),
  ("V003", 2, "P02",  95, "2024-01-06"), ("V004", 2, "P08",  42, "2024-01-06"),
  ("V005", 3, "P03",  80, "2024-01-08"), ("V006", 3, "P07",  25, "2024-01-08"),
  ("V007", 4, "P04",  60, "2024-01-10"), ("V008", 4, "P10",  15, "2024-01-10"),
  ("V009", 5, "P05", 200, "2024-01-12"), ("V010", 5, "P09",  70, "2024-01-12"),
  ("V011", 6, "P01",  90, "2024-01-15"), ("V012", 6, "P06",  22, "2024-01-15"),
  ("V013", 1, "P03",  55, "2024-02-01"), ("V014", 1, "P07",  10, "2024-02-01"),
  ("V015", 2, "P05", 180, "2024-02-03"), ("V016", 2, "P10",  20, "2024-02-03"),
  ("V017", 3, "P01", 110, "2024-02-07"), ("V018", 3, "P08",  35, "2024-02-07"),
  ("V019", 4, "P02",  75, "2024-02-10"), ("V020", 4, "P06",  30, "2024-02-10"),
  ("V021", 5, "P03",  65, "2024-02-14"), ("V022", 5, "P07",  12, "2024-02-14"),
  ("V023", 6, "P04",  88, "2024-02-18"), ("V024", 6, "P09",  50, "2024-02-18"),
  ("V025", 1, "P10",  25, "2024-03-02"), ("V026", 1, "P02", 100, "2024-03-02"),
  ("V027", 2, "P03",  70, "2024-03-05"), ("V028", 2, "P06",  28, "2024-03-05"),
  ("V029", 3, "P05", 220, "2024-03-08"), ("V030", 3, "P07",  15, "2024-03-08"),
  ("V031", 4, "P01",  95, "2024-03-12"), ("V032", 4, "P08",  48, "2024-03-12"),
  ("V033", 5, "P02",  85, "2024-03-15"), ("V034", 5, "P10",  18, "2024-03-15"),
  ("V035", 6, "P03",  72, "2024-03-20"), ("V036", 6, "P06",  35, "2024-03-20")
).toDF("ventaId", "tiendaId", "productoId", "unidades", "fecha")

// Añadir importe (unidades × precioUnitario)
val ventasConImporte = ventasFM
  .join(productosFM.select("productoId", "precioUnitario"),
        ventasFM("productoId") === productosFM("productoId"), "inner")
  .withColumn("importe", (col("unidades") * col("precioUnitario")).cast("double"))
  .drop(productosFM("productoId"))
  .drop("precioUnitario")

println(s"Datos cargados: ${ventasConImporte.count()} ventas con importe ✅")

Datos cargados: 36 ventas con importe ✅


tiendas: org.apache.spark.sql.package.DataFrame = [tiendaId: int, nombre: string ... 2 more fields]
productosFM: org.apache.spark.sql.package.DataFrame = [productoId: string, nombreProducto: string ... 2 more fields]
ventasFM: org.apache.spark.sql.package.DataFrame = [ventaId: string, tiendaId: int ... 3 more fields]
ventasConImporte: org.apache.spark.sql.package.DataFrame = [ventaId: string, tiendaId: int ... 4 more fields]

### 🔹 Celda 3 — Registrar las vistas temporales

In [22]:
tiendas.createOrReplaceTempView("tiendas")
productosFM.createOrReplaceTempView("productos")
ventasConImporte.createOrReplaceTempView("ventas")

println("Vistas registradas:")
spark.catalog.listTables().show()

Vistas registradas:
+---------+-------+---------+-----------+---------+-----------+
|     name|catalog|namespace|description|tableType|isTemporary|
+---------+-------+---------+-----------+---------+-----------+
|productos|   NULL|       []|       NULL|TEMPORARY|       true|
|  tiendas|   NULL|       []|       NULL|TEMPORARY|       true|
|   ventas|   NULL|       []|       NULL|TEMPORARY|       true|
+---------+-------+---------+-----------+---------+-----------+



### 🔹 Celda 4 — Ventas por categoría: SQL vs API

In [23]:
val resultadoSQL = spark.sql("""
  SELECT
    p.categoria,
    COUNT(*)                       AS num_ventas,
    SUM(v.unidades)                AS total_unidades,
    ROUND(SUM(v.importe), 2)       AS total_importe
  FROM ventas v
  JOIN productos p ON v.productoId = p.productoId
  GROUP BY p.categoria
  ORDER BY total_importe DESC
""")

println("=== SQL ===")
resultadoSQL.show()

val resultadoAPI = ventasConImporte
  .join(productosFM.select("productoId", "categoria"),
        ventasConImporte("productoId") === productosFM("productoId"), "inner")
  .groupBy("categoria")
  .agg(
    count("*").alias("num_ventas"),
    sum("unidades").alias("total_unidades"),
    round(sum("importe"), 2).alias("total_importe")
  )
  .orderBy(col("total_importe").desc)

println("=== DataFrame API ===")
resultadoAPI.show()
println(s"Filas SQL: ${resultadoSQL.count()} | Filas API: ${resultadoAPI.count()}")

=== SQL ===
+----------+----------+--------------+-------------+
| categoria|num_ventas|total_unidades|total_importe|
+----------+----------+--------------+-------------+
|Carnicería|         9|           195|       1506.7|
|   Lácteos|         9|           757|       984.95|
|   Bebidas|         5|           748|        492.0|
|   Aceites|         4|            78|        452.4|
| Panadería|         4|           355|       443.75|
|    Frutas|         3|           125|        275.0|
|  Verduras|         2|           120|        234.0|
+----------+----------+--------------+-------------+

=== DataFrame API ===
+----------+----------+--------------+-------------+
| categoria|num_ventas|total_unidades|total_importe|
+----------+----------+--------------+-------------+
|Carnicería|         9|           195|       1506.7|
|   Lácteos|         9|           757|       984.95|
|   Bebidas|         5|           748|        492.0|
|   Aceites|         4|            78|        452.4|
| Panadería

resultadoSQL: org.apache.spark.sql.package.DataFrame = [categoria: string, num_ventas: bigint ... 2 more fields]
resultadoAPI: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [categoria: string, num_ventas: bigint ... 2 more fields]

### 🔹 Celda 5 — Subconsulta `IN`: ventas región Centro

In [24]:
val tiendasCentro = spark.sql("""
  SELECT ventaId, tiendaId, productoId, unidades, importe, fecha
  FROM ventas
  WHERE tiendaId IN (
    SELECT tiendaId FROM tiendas WHERE region = 'Centro'
  )
  ORDER BY fecha, tiendaId
""")

println("=== IN ===")
tiendasCentro.show()
println(s"Total: ${tiendasCentro.count()}")

=== IN ===
+-------+--------+----------+--------+------------------+----------+
|ventaId|tiendaId|productoId|unidades|           importe|     fecha|
+-------+--------+----------+--------+------------------+----------+
|   V001|       1|       P01|     120|             106.8|2024-01-05|
|   V002|       1|       P06|      18|             124.2|2024-01-05|
|   V003|       2|       P02|      95|            118.75|2024-01-06|
|   V004|       2|       P08|      42|              92.4|2024-01-06|
|   V014|       1|       P07|      10|              95.0|2024-02-01|
|   V013|       1|       P03|      55|              99.0|2024-02-01|
|   V015|       2|       P05|     180|              81.0|2024-02-03|
|   V016|       2|       P10|      20|             116.0|2024-02-03|
|   V025|       1|       P10|      25|             145.0|2024-03-02|
|   V026|       1|       P02|     100|             125.0|2024-03-02|
|   V028|       2|       P06|      28|193.20000000000002|2024-03-05|
|   V027|       2|     

tiendasCentro: org.apache.spark.sql.package.DataFrame = [ventaId: string, tiendaId: int ... 4 more fields]

### 🔹 Celda 6 — Subconsulta `EXISTS`: tiendas que vendieron Carnicería

In [25]:
val tiendasConCarne = spark.sql("""
  SELECT t.tiendaId, t.nombre, t.ciudad
  FROM tiendas t
  WHERE EXISTS (
    SELECT 1
    FROM ventas v
    JOIN productos p ON v.productoId = p.productoId
    WHERE v.tiendaId = t.tiendaId AND p.categoria = 'Carnicería'
  )
  ORDER BY t.ciudad
""")

println("=== EXISTS ===")
tiendasConCarne.show()

=== EXISTS ===
+--------+--------------------+---------+
|tiendaId|              nombre|   ciudad|
+--------+--------------------+---------+
|       3|  FreshMart Diagonal|Barcelona|
|       4|    FreshMart Gracia|Barcelona|
|       5|   FreshMart Nervión|   Bilbao|
|       6|FreshMart Casco V...|   Bilbao|
|       1|FreshMart Castellana|   Madrid|
|       2|  FreshMart Vallecas|   Madrid|
+--------+--------------------+---------+



tiendasConCarne: org.apache.spark.sql.package.DataFrame = [tiendaId: int, nombre: string ... 1 more field]

### 🔹 Celda 7 — Subconsulta derivada en `FROM`: media diaria por región

In [26]:
val mediaDiariaPorRegion = spark.sql("""
  SELECT region, ROUND(AVG(total_dia), 2) AS media_importe_diario
  FROM (
    SELECT t.region, v.fecha, SUM(v.importe) AS total_dia
    FROM ventas v
    JOIN tiendas t ON v.tiendaId = t.tiendaId
    GROUP BY t.region, v.fecha
  ) totales_diarios
  GROUP BY region
  ORDER BY media_importe_diario DESC
""")

println("=== Subconsulta en FROM ===")
mediaDiariaPorRegion.show()

=== Subconsulta en FROM ===
+------+--------------------+
|region|media_importe_diario|
+------+--------------------+
| Norte|              250.11|
|  Este|               244.3|
|Centro|              237.06|
+------+--------------------+



mediaDiariaPorRegion: org.apache.spark.sql.package.DataFrame = [region: string, media_importe_diario: double]

### 🔹 Celda 8 — CTE simple: ventas sobre el ticket medio global

In [27]:
val ventasSobreMedia = spark.sql("""
  WITH media_global AS (
    SELECT AVG(importe) AS media FROM ventas
  )
  SELECT
    v.ventaId,
    v.tiendaId,
    v.productoId,
    ROUND(v.importe, 2)             AS importe,
    ROUND(mg.media, 2)              AS media_global,
    ROUND(v.importe - mg.media, 2)  AS diferencia
  FROM ventas v
  CROSS JOIN media_global mg
  WHERE v.importe > mg.media
  ORDER BY diferencia DESC
""")

println("=== CTE simple ===")
ventasSobreMedia.show()
println(s"Ventas sobre la media: ${ventasSobreMedia.count()}")

=== CTE simple ===
+-------+--------+----------+-------+------------+----------+
|ventaId|tiendaId|productoId|importe|media_global|diferencia|
+-------+--------+----------+-------+------------+----------+
|   V036|       6|       P06|  241.5|      121.91|    119.59|
|   V006|       3|       P07|  237.5|      121.91|    115.59|
|   V020|       4|       P06|  207.0|      121.91|     85.09|
|   V028|       2|       P06|  193.2|      121.91|     71.29|
|   V012|       6|       P06|  151.8|      121.91|     29.89|
|   V025|       1|       P10|  145.0|      121.91|     23.09|
|   V005|       3|       P03|  144.0|      121.91|     22.09|
|   V030|       3|       P07|  142.5|      121.91|     20.59|
|   V010|       5|       P09|  136.5|      121.91|     14.59|
|   V023|       6|       P04|  132.0|      121.91|     10.09|
|   V035|       6|       P03|  129.6|      121.91|      7.69|
|   V027|       2|       P03|  126.0|      121.91|      4.09|
|   V026|       1|       P02|  125.0|      121.91| 

ventasSobreMedia: org.apache.spark.sql.package.DataFrame = [ventaId: string, tiendaId: int ... 4 more fields]

### 🔹 Celda 9 — CTE encadenada: informe ejecutivo por región y categoría

In [28]:
val informeEjecutivo = spark.sql("""
  WITH ventas_enriquecidas AS (
    SELECT
      t.region,
      t.nombre AS tienda,
      p.categoria,
      v.unidades,
      v.importe
    FROM ventas v
    JOIN tiendas   t ON v.tiendaId   = t.tiendaId
    JOIN productos p ON v.productoId = p.productoId
  ),
  resumen_por_region AS (
    SELECT
      region,
      categoria,
      SUM(unidades)          AS total_unidades,
      ROUND(SUM(importe), 2) AS total_importe,
      COUNT(*)               AS num_transacciones
    FROM ventas_enriquecidas
    GROUP BY region, categoria
  )
  SELECT
    region,
    categoria,
    total_unidades,
    total_importe,
    num_transacciones,
    ROUND(total_importe / num_transacciones, 2) AS ticket_medio
  FROM resumen_por_region
  ORDER BY region, total_importe DESC
""")

println("=== CTE encadenada ===")
informeEjecutivo.show(30, truncate = false)

=== CTE encadenada ===
+------+----------+--------------+-------------+-----------------+------------+
|region|categoria |total_unidades|total_importe|num_transacciones|ticket_medio|
+------+----------+--------------+-------------+-----------------+------------+
|Centro|Carnicería|56            |412.4        |3                |137.47      |
|Centro|Lácteos   |245           |331.8        |3                |110.6       |
|Centro|Aceites   |45            |261.0        |2                |130.5       |
|Centro|Panadería |195           |243.75       |2                |121.88      |
|Centro|Frutas    |42            |92.4         |1                |92.4        |
|Centro|Bebidas   |180           |81.0         |1                |81.0        |
|Este  |Carnicería|70            |587.0        |3                |195.67      |
|Este  |Lácteos   |285           |326.45       |3                |108.82      |
|Este  |Bebidas   |280           |189.0        |2                |94.5        |
|Este  |Frutas   

informeEjecutivo: org.apache.spark.sql.package.DataFrame = [region: string, categoria: string ... 4 more fields]

### 🔹 Celda 10 — Explorar el catálogo y gestionar vistas

In [29]:
println("Vistas en el catálogo:")
spark.catalog.listTables().show()

println(s"¿Existe 'ventas'?    ${spark.catalog.tableExists("ventas")}")
println(s"¿Existe 'empleados'? ${spark.catalog.tableExists("empleados")}")

spark.sql("""
  SELECT tiendaId, ROUND(SUM(importe), 2) AS total_trimestre
  FROM ventas
  GROUP BY tiendaId
""").createOrReplaceTempView("resumen_tiendas")

println("\nVistas tras añadir 'resumen_tiendas':")
spark.catalog.listTables().show()

spark.sql("""
  SELECT t.nombre, t.ciudad, r.total_trimestre
  FROM resumen_tiendas r
  JOIN tiendas t ON r.tiendaId = t.tiendaId
  ORDER BY r.total_trimestre DESC
""").show()

spark.catalog.dropTempView("resumen_tiendas")
println(s"¿Sigue existiendo 'resumen_tiendas'? ${spark.catalog.tableExists("resumen_tiendas")}")

Vistas en el catálogo:
+---------+-------+---------+-----------+---------+-----------+
|     name|catalog|namespace|description|tableType|isTemporary|
+---------+-------+---------+-----------+---------+-----------+
|productos|   NULL|       []|       NULL|TEMPORARY|       true|
|  tiendas|   NULL|       []|       NULL|TEMPORARY|       true|
|   ventas|   NULL|       []|       NULL|TEMPORARY|       true|
+---------+-------+---------+-----------+---------+-----------+

¿Existe 'ventas'?    true
¿Existe 'empleados'? false

Vistas tras añadir 'resumen_tiendas':
+---------------+-------+---------+-----------+---------+-----------+
|           name|catalog|namespace|description|tableType|isTemporary|
+---------------+-------+---------+-----------+---------+-----------+
|      productos|   NULL|       []|       NULL|TEMPORARY|       true|
|resumen_tiendas|   NULL|       []|       NULL|TEMPORARY|       true|
|        tiendas|   NULL|       []|       NULL|TEMPORARY|       true|
|         ventas

res29_8: Boolean = true

### 🔹 Celda 11 — Verificación final FreshMart

In [30]:
println("=" * 55)
println("RESUMEN — Sesión 2 | FreshMart")
println("=" * 55)

val checks = Seq(
  ("Vistas registradas en el catálogo",
    spark.catalog.tableExists("ventas") &&
    spark.catalog.tableExists("tiendas") &&
    spark.catalog.tableExists("productos")),
  ("SQL y API producen el mismo número de filas",
    resultadoSQL.count() == resultadoAPI.count()),
  ("Subconsulta IN — ventas región Centro",  tiendasCentro.count()  > 0),
  ("EXISTS — tiendas con ventas Carnicería", tiendasConCarne.count() > 0),
  ("CTE encadenada — informe ejecutivo",     informeEjecutivo.count() > 0)
)

checks.foreach { case (desc, ok) =>
  println(s"${if (ok) "✅ CORRECTO" else "❌ REVISAR"} — $desc")
}

RESUMEN — Sesión 2 | FreshMart
✅ CORRECTO — Vistas registradas en el catálogo
✅ CORRECTO — SQL y API producen el mismo número de filas
✅ CORRECTO — Subconsulta IN — ventas región Centro
✅ CORRECTO — EXISTS — tiendas con ventas Carnicería
✅ CORRECTO — CTE encadenada — informe ejecutivo


checks: Seq[(String, Boolean)] = List(
  ("Vistas registradas en el catálogo", true),
  ("SQL y API producen el mismo número de filas", true),
  ("Subconsulta IN — ventas región Centro", true),
  ("EXISTS — tiendas con ventas Carnicería", true),
  ("CTE encadenada — informe ejecutivo", true)
)

---

## 🏢 Caso de Estudio — *EduTrack Academy*

Plataforma de formación online. Tres fuentes: `estudiantes`, `cursos`, `matriculas`.

> Antes de registrar las nuevas vistas, eliminamos las anteriores para evitar colisiones de nombre.


In [31]:
// Limpieza de vistas previas (de FreshMart)
List("ventas", "tiendas", "productos").foreach { v =>
  if (spark.catalog.tableExists(v)) spark.catalog.dropTempView(v)
}
println("Vistas FreshMart eliminadas. Catálogo:")
spark.catalog.listTables().show()

Vistas FreshMart eliminadas. Catálogo:
+----+-------+---------+-----------+---------+-----------+
|name|catalog|namespace|description|tableType|isTemporary|
+----+-------+---------+-----------+---------+-----------+
+----+-------+---------+-----------+---------+-----------+



### Celda 2 — Datos de EduTrack Academy

In [32]:
val estudiantes = Seq(
  (1,  "Lucía Fernández",  "Madrid",    "2023-09-01", "Premium"),
  (2,  "Carlos Romero",    "Barcelona", "2023-09-15", "Basic"),
  (3,  "Ana Torres",       "Sevilla",   "2023-10-01", "Premium"),
  (4,  "David Iglesias",   "Madrid",    "2023-10-10", "Basic"),
  (5,  "Marta Soler",      "Valencia",  "2023-11-01", "Premium"),
  (6,  "Pedro Navarro",    "Bilbao",    "2023-11-20", "Basic"),
  (7,  "Elena Vidal",      "Madrid",    "2024-01-05", "Premium"),
  (8,  "Javier Molina",    "Barcelona", "2024-01-15", "Basic"),
  (9,  "Sara Castillo",    "Sevilla",   "2024-02-01", "Premium"),
  (10, "Rubén Ortega",     "Madrid",    "2024-02-20", "Basic")
).toDF("estudianteId", "nombre", "ciudad", "fechaAlta", "plan")

val cursos = Seq(
  ("C01", "Python para Data Science",    "Programación",   299.0, "Avanzado"),
  ("C02", "SQL desde cero",              "Bases de Datos", 149.0, "Básico"),
  ("C03", "Machine Learning con Python", "IA",             399.0, "Avanzado"),
  ("C04", "Scala Funcional",             "Programación",   249.0, "Intermedio"),
  ("C05", "Apache Spark",                "Big Data",       349.0, "Avanzado"),
  ("C06", "Power BI para negocios",      "Analítica",      199.0, "Básico"),
  ("C07", "Docker y Kubernetes",         "DevOps",         279.0, "Intermedio"),
  ("C08", "Estadística para Data",       "Analítica",     179.0, "Básico")
).toDF("cursoId", "nombreCurso", "area", "precio", "nivel")

val matriculas = Seq(
  (1,  1,  "C01", "2023-09-05", 299.0, "Completado"),
  (2,  1,  "C04", "2023-10-01", 249.0, "Completado"),
  (3,  2,  "C02", "2023-09-20", 149.0, "Completado"),
  (4,  2,  "C06", "2023-11-01", 199.0, "En progreso"),
  (5,  3,  "C01", "2023-10-05", 299.0, "Completado"),
  (6,  3,  "C03", "2023-11-10", 399.0, "En progreso"),
  (7,  3,  "C05", "2024-01-15", 349.0, "En progreso"),
  (8,  4,  "C02", "2023-10-15", 149.0, "Completado"),
  (9,  5,  "C03", "2023-11-05", 399.0, "Completado"),
  (10, 5,  "C05", "2023-12-01", 349.0, "Completado"),
  (11, 5,  "C07", "2024-01-20", 279.0, "En progreso"),
  (12, 6,  "C06", "2023-12-01", 199.0, "Completado"),
  (13, 7,  "C01", "2024-01-10", 299.0, "En progreso"),
  (14, 7,  "C04", "2024-02-01", 249.0, "En progreso"),
  (15, 7,  "C05", "2024-02-15", 349.0, "En progreso"),
  (16, 8,  "C02", "2024-01-20", 149.0, "En progreso"),
  (17, 9,  "C01", "2024-02-05", 299.0, "En progreso"),
  (18, 9,  "C08", "2024-02-10", 179.0, "En progreso"),
  (19, 10, "C06", "2024-02-25", 199.0, "En progreso")
).toDF("matriculaId", "estudianteId", "cursoId", "fechaMatricula", "importePagado", "estado")

println(s"estudiantes: ${estudiantes.count()} | cursos: ${cursos.count()} | matriculas: ${matriculas.count()}")

estudiantes: 10 | cursos: 8 | matriculas: 19


estudiantes: org.apache.spark.sql.package.DataFrame = [estudianteId: int, nombre: string ... 3 more fields]
cursos: org.apache.spark.sql.package.DataFrame = [cursoId: string, nombreCurso: string ... 3 more fields]
matriculas: org.apache.spark.sql.package.DataFrame = [matriculaId: int, estudianteId: int ... 4 more fields]

### Celda 3 — Registrar las vistas y verificar el catálogo

In [33]:
estudiantes.createOrReplaceTempView("estudiantes")
cursos.createOrReplaceTempView("cursos")
matriculas.createOrReplaceTempView("matriculas")

println("✅ Vistas registradas:")
spark.catalog.listTables().show()

✅ Vistas registradas:
+-----------+-------+---------+-----------+---------+-----------+
|       name|catalog|namespace|description|tableType|isTemporary|
+-----------+-------+---------+-----------+---------+-----------+
|     cursos|   NULL|       []|       NULL|TEMPORARY|       true|
|estudiantes|   NULL|       []|       NULL|TEMPORARY|       true|
| matriculas|   NULL|       []|       NULL|TEMPORARY|       true|
+-----------+-------+---------+-----------+---------+-----------+



### 📋 Consulta 1 — Ingresos totales por área temática (SQL + API)

In [34]:
val ingresosPorAreaSQL = spark.sql("""
  SELECT
    c.area,
    COUNT(*)                       AS num_matriculas,
    ROUND(SUM(m.importePagado), 2) AS ingresos_totales,
    ROUND(AVG(m.importePagado), 2) AS precio_medio
  FROM matriculas m
  JOIN cursos c ON m.cursoId = c.cursoId
  GROUP BY c.area
  ORDER BY ingresos_totales DESC
""")

println("=== SQL ===")
ingresosPorAreaSQL.show()

val ingresosPorAreaAPI = matriculas
  .join(cursos.select("cursoId", "area"),
        matriculas("cursoId") === cursos("cursoId"), "inner")
  .groupBy("area")
  .agg(
    count("*").alias("num_matriculas"),
    round(sum("importePagado"), 2).alias("ingresos_totales"),
    round(avg("importePagado"), 2).alias("precio_medio")
  )
  .orderBy(col("ingresos_totales").desc)

println("=== DataFrame API ===")
ingresosPorAreaAPI.show()
println(s"Filas SQL: ${ingresosPorAreaSQL.count()} | API: ${ingresosPorAreaAPI.count()}")

=== SQL ===
+--------------+--------------+----------------+------------+
|          area|num_matriculas|ingresos_totales|precio_medio|
+--------------+--------------+----------------+------------+
|  Programación|             6|          1694.0|      282.33|
|      Big Data|             3|          1047.0|       349.0|
|            IA|             2|           798.0|       399.0|
|     Analítica|             4|           776.0|       194.0|
|Bases de Datos|             3|           447.0|       149.0|
|        DevOps|             1|           279.0|       279.0|
+--------------+--------------+----------------+------------+

=== DataFrame API ===
+--------------+--------------+----------------+------------+
|          area|num_matriculas|ingresos_totales|precio_medio|
+--------------+--------------+----------------+------------+
|  Programación|             6|          1694.0|      282.33|
|      Big Data|             3|          1047.0|       349.0|
|            IA|             2|    

ingresosPorAreaSQL: org.apache.spark.sql.package.DataFrame = [area: string, num_matriculas: bigint ... 2 more fields]
ingresosPorAreaAPI: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [area: string, num_matriculas: bigint ... 2 more fields]

### 📋 Consulta 2 — Subconsulta `IN`: matrículas Premium

In [35]:
val matriculasPremium = spark.sql("""
  SELECT m.matriculaId, m.estudianteId, m.cursoId, m.fechaMatricula, m.importePagado, m.estado
  FROM matriculas m
  WHERE m.estudianteId IN (
    SELECT estudianteId FROM estudiantes WHERE plan = 'Premium'
  )
  ORDER BY m.estudianteId, m.fechaMatricula
""")

println("=== IN — Premium ===")
matriculasPremium.show()
println(s"Total matrículas Premium: ${matriculasPremium.count()}")

=== IN — Premium ===
+-----------+------------+-------+--------------+-------------+-----------+
|matriculaId|estudianteId|cursoId|fechaMatricula|importePagado|     estado|
+-----------+------------+-------+--------------+-------------+-----------+
|          1|           1|    C01|    2023-09-05|        299.0| Completado|
|          2|           1|    C04|    2023-10-01|        249.0| Completado|
|          5|           3|    C01|    2023-10-05|        299.0| Completado|
|          6|           3|    C03|    2023-11-10|        399.0|En progreso|
|          7|           3|    C05|    2024-01-15|        349.0|En progreso|
|          9|           5|    C03|    2023-11-05|        399.0| Completado|
|         10|           5|    C05|    2023-12-01|        349.0| Completado|
|         11|           5|    C07|    2024-01-20|        279.0|En progreso|
|         13|           7|    C01|    2024-01-10|        299.0|En progreso|
|         14|           7|    C04|    2024-02-01|        249.0|En p

matriculasPremium: org.apache.spark.sql.package.DataFrame = [matriculaId: int, estudianteId: int ... 4 more fields]

### 📋 Consulta 3 — Subconsulta `EXISTS`: cursos con al menos un completado

In [36]:
val cursosConCompletados = spark.sql("""
  SELECT c.cursoId, c.nombreCurso, c.area, c.nivel, c.precio
  FROM cursos c
  WHERE EXISTS (
    SELECT 1 FROM matriculas m
    WHERE m.cursoId = c.cursoId AND m.estado = 'Completado'
  )
  ORDER BY c.area, c.precio DESC
""")

println("=== EXISTS ===")
cursosConCompletados.show(truncate = false)
println(s"Cursos con completados: ${cursosConCompletados.count()} de ${cursos.count()}")

=== EXISTS ===
+-------+---------------------------+--------------+----------+------+
|cursoId|nombreCurso                |area          |nivel     |precio|
+-------+---------------------------+--------------+----------+------+
|C06    |Power BI para negocios     |Analítica     |Básico    |199.0 |
|C02    |SQL desde cero             |Bases de Datos|Básico    |149.0 |
|C05    |Apache Spark               |Big Data      |Avanzado  |349.0 |
|C03    |Machine Learning con Python|IA            |Avanzado  |399.0 |
|C01    |Python para Data Science   |Programación  |Avanzado  |299.0 |
|C04    |Scala Funcional            |Programación  |Intermedio|249.0 |
+-------+---------------------------+--------------+----------+------+

Cursos con completados: 6 de 8


cursosConCompletados: org.apache.spark.sql.package.DataFrame = [cursoId: string, nombreCurso: string ... 3 more fields]

### 📋 Consulta 4 — Subconsulta derivada en `FROM`: ranking de gasto

In [37]:
val rankingGasto = spark.sql("""
  SELECT
    e.nombre,
    e.ciudad,
    e.plan,
    gasto.total_gastado,
    gasto.num_cursos
  FROM (
    SELECT
      estudianteId,
      ROUND(SUM(importePagado), 2) AS total_gastado,
      COUNT(*)                     AS num_cursos
    FROM matriculas
    GROUP BY estudianteId
  ) gasto
  JOIN estudiantes e ON gasto.estudianteId = e.estudianteId
  ORDER BY gasto.total_gastado DESC
""")

println("=== Ranking gasto ===")
rankingGasto.show(truncate = false)

=== Ranking gasto ===
+---------------+---------+-------+-------------+----------+
|nombre         |ciudad   |plan   |total_gastado|num_cursos|
+---------------+---------+-------+-------------+----------+
|Ana Torres     |Sevilla  |Premium|1047.0       |3         |
|Marta Soler    |Valencia |Premium|1027.0       |3         |
|Elena Vidal    |Madrid   |Premium|897.0        |3         |
|Lucía Fernández|Madrid   |Premium|548.0        |2         |
|Sara Castillo  |Sevilla  |Premium|478.0        |2         |
|Carlos Romero  |Barcelona|Basic  |348.0        |2         |
|Rubén Ortega   |Madrid   |Basic  |199.0        |1         |
|Pedro Navarro  |Bilbao   |Basic  |199.0        |1         |
|David Iglesias |Madrid   |Basic  |149.0        |1         |
|Javier Molina  |Barcelona|Basic  |149.0        |1         |
+---------------+---------+-------+-------------+----------+



rankingGasto: org.apache.spark.sql.package.DataFrame = [nombre: string, ciudad: string ... 3 more fields]

### 📋 Consulta 5 — Subconsulta escalar en `SELECT`: desviación respecto al precio medio

In [38]:
val desviacionPrecio = spark.sql("""
  SELECT
    m.matriculaId,
    e.nombre AS estudiante,
    c.nombreCurso,
    ROUND(m.importePagado, 2)                                      AS importe,
    ROUND((SELECT AVG(importePagado) FROM matriculas), 2)          AS media_global,
    ROUND(m.importePagado - (SELECT AVG(importePagado) FROM matriculas), 2) AS diferencia
  FROM matriculas m
  JOIN estudiantes e ON m.estudianteId = e.estudianteId
  JOIN cursos      c ON m.cursoId      = c.cursoId
  ORDER BY diferencia DESC
""")

println("=== Subconsulta escalar ===")
desviacionPrecio.show(20, truncate = false)

=== Subconsulta escalar ===
+-----------+---------------+---------------------------+-------+------------+----------+
|matriculaId|estudiante     |nombreCurso                |importe|media_global|diferencia|
+-----------+---------------+---------------------------+-------+------------+----------+
|6          |Ana Torres     |Machine Learning con Python|399.0  |265.32      |133.68    |
|9          |Marta Soler    |Machine Learning con Python|399.0  |265.32      |133.68    |
|15         |Elena Vidal    |Apache Spark               |349.0  |265.32      |83.68     |
|7          |Ana Torres     |Apache Spark               |349.0  |265.32      |83.68     |
|10         |Marta Soler    |Apache Spark               |349.0  |265.32      |83.68     |
|13         |Elena Vidal    |Python para Data Science   |299.0  |265.32      |33.68     |
|17         |Sara Castillo  |Python para Data Science   |299.0  |265.32      |33.68     |
|1          |Lucía Fernández|Python para Data Science   |299.0  |265.32 

desviacionPrecio: org.apache.spark.sql.package.DataFrame = [matriculaId: int, estudiante: string ... 4 more fields]

### 📋 Consulta 6 — CTE simple: estudiantes sin completados

In [39]:
val sinCompletados = spark.sql("""
  WITH estudiantes_con_completados AS (
    SELECT DISTINCT estudianteId FROM matriculas WHERE estado = 'Completado'
  )
  SELECT e.estudianteId, e.nombre, e.ciudad, e.plan, e.fechaAlta
  FROM estudiantes e
  WHERE e.estudianteId NOT IN (SELECT estudianteId FROM estudiantes_con_completados)
  ORDER BY e.fechaAlta
""")

println("=== CTE simple ===")
sinCompletados.show(truncate = false)
println(s"Estudiantes sin completados: ${sinCompletados.count()}")

=== CTE simple ===
+------------+-------------+---------+-------+----------+
|estudianteId|nombre       |ciudad   |plan   |fechaAlta |
+------------+-------------+---------+-------+----------+
|7           |Elena Vidal  |Madrid   |Premium|2024-01-05|
|8           |Javier Molina|Barcelona|Basic  |2024-01-15|
|9           |Sara Castillo|Sevilla  |Premium|2024-02-01|
|10          |Rubén Ortega |Madrid   |Basic  |2024-02-20|
+------------+-------------+---------+-------+----------+

Estudiantes sin completados: 4


sinCompletados: org.apache.spark.sql.package.DataFrame = [estudianteId: int, nombre: string ... 3 more fields]

### 📋 Consulta 7 — CTE encadenada: rendimiento por plan de suscripción

In [40]:
val rendimientoPorPlan = spark.sql("""
  WITH gasto_por_estudiante AS (
    SELECT
      m.estudianteId,
      COUNT(*) AS total_cursos,
      SUM(CASE WHEN m.estado = 'Completado' THEN 1 ELSE 0 END) AS cursos_completados,
      ROUND(SUM(m.importePagado), 2) AS gasto_total
    FROM matriculas m
    GROUP BY m.estudianteId
  ),
  gasto_enriquecido AS (
    SELECT e.plan, g.total_cursos, g.cursos_completados, g.gasto_total
    FROM gasto_por_estudiante g
    JOIN estudiantes e ON g.estudianteId = e.estudianteId
  ),
  resumen_por_plan AS (
    SELECT
      plan,
      COUNT(*)                          AS num_estudiantes,
      ROUND(AVG(gasto_total), 2)        AS gasto_medio_por_persona,
      ROUND(AVG(total_cursos), 2)       AS cursos_medios_por_persona,
      ROUND(AVG(cursos_completados), 2) AS completados_medios,
      ROUND(SUM(gasto_total), 2)        AS ingresos_totales
    FROM gasto_enriquecido
    GROUP BY plan
  )
  SELECT
    plan,
    num_estudiantes,
    ingresos_totales,
    gasto_medio_por_persona,
    cursos_medios_por_persona,
    completados_medios,
    ROUND(completados_medios / cursos_medios_por_persona * 100, 1) AS tasa_completado_pct
  FROM resumen_por_plan
  ORDER BY ingresos_totales DESC
""")

println("=== CTE encadenada ===")
rendimientoPorPlan.show(truncate = false)

=== CTE encadenada ===
+-------+---------------+----------------+-----------------------+-------------------------+------------------+-------------------+
|plan   |num_estudiantes|ingresos_totales|gasto_medio_por_persona|cursos_medios_por_persona|completados_medios|tasa_completado_pct|
+-------+---------------+----------------+-----------------------+-------------------------+------------------+-------------------+
|Premium|5              |3997.0          |799.4                  |2.6                      |1.0               |38.5               |
|Basic  |5              |1044.0          |208.8                  |1.2                      |0.6               |50.0               |
+-------+---------------+----------------+-----------------------+-------------------------+------------------+-------------------+



rendimientoPorPlan: org.apache.spark.sql.package.DataFrame = [plan: string, num_estudiantes: bigint ... 5 more fields]

### 📋 Consulta 8 — Vista derivada y consulta sobre ella

In [41]:
spark.sql("""
  SELECT
    c.cursoId,
    c.nombreCurso,
    c.area,
    c.nivel,
    c.precio                       AS precio_catalogo,
    COUNT(m.matriculaId)           AS total_matriculas,
    ROUND(SUM(m.importePagado), 2) AS ingresos_reales,
    SUM(CASE WHEN m.estado = 'Completado' THEN 1 ELSE 0 END) AS completados
  FROM cursos c
  LEFT JOIN matriculas m ON c.cursoId = m.cursoId
  GROUP BY c.cursoId, c.nombreCurso, c.area, c.nivel, c.precio
""").createOrReplaceTempView("resumen_cursos")

println("Vistas tras crear 'resumen_cursos':")
spark.catalog.listTables().show()

val cursosRentables = spark.sql("""
  SELECT nombreCurso, area, total_matriculas, ingresos_reales, completados
  FROM resumen_cursos
  WHERE total_matriculas >= 3
  ORDER BY ingresos_reales DESC
""")

println("=== Cursos con 3+ matrículas ===")
cursosRentables.show(truncate = false)

Vistas tras crear 'resumen_cursos':
+--------------+-------+---------+-----------+---------+-----------+
|          name|catalog|namespace|description|tableType|isTemporary|
+--------------+-------+---------+-----------+---------+-----------+
|        cursos|   NULL|       []|       NULL|TEMPORARY|       true|
|   estudiantes|   NULL|       []|       NULL|TEMPORARY|       true|
|    matriculas|   NULL|       []|       NULL|TEMPORARY|       true|
|resumen_cursos|   NULL|       []|       NULL|TEMPORARY|       true|
+--------------+-------+---------+-----------+---------+-----------+

=== Cursos con 3+ matrículas ===
+------------------------+--------------+----------------+---------------+-----------+
|nombreCurso             |area          |total_matriculas|ingresos_reales|completados|
+------------------------+--------------+----------------+---------------+-----------+
|Python para Data Science|Programación  |4               |1196.0         |2          |
|Apache Spark            |Big 

cursosRentables: org.apache.spark.sql.package.DataFrame = [nombreCurso: string, area: string ... 3 more fields]

### Celda final — Limpieza y verificación EduTrack

In [42]:
spark.catalog.dropTempView("resumen_cursos")

println("Catálogo final:")
spark.catalog.listTables().show()

println("\n" + "=" * 58)
println("RESUMEN — Caso de Estudio EduTrack Academy")
println("=" * 58)

val checks = Seq(
  ("Vistas base registradas (3)",
    spark.catalog.tableExists("estudiantes") &&
    spark.catalog.tableExists("cursos") &&
    spark.catalog.tableExists("matriculas")),
  ("SQL y API producen el mismo resultado",
    ingresosPorAreaSQL.count() == ingresosPorAreaAPI.count()),
  ("IN — matrículas Premium",        matriculasPremium.count()    == 13),
  ("EXISTS — cursos con completados", cursosConCompletados.count() == 6),
  ("FROM — ranking de gasto",         rankingGasto.count()         == 10),
  ("CTE simple — sin completados",    sinCompletados.count()       == 3),
  ("CTE encadenada — por plan",       rendimientoPorPlan.count()   == 2),
  ("Vista derivada eliminada",        !spark.catalog.tableExists("resumen_cursos"))
)

checks.foreach { case (desc, ok) =>
  println(s"${if (ok) "✅ CORRECTO" else "❌ REVISAR"} — $desc")
}

Catálogo final:
+-----------+-------+---------+-----------+---------+-----------+
|       name|catalog|namespace|description|tableType|isTemporary|
+-----------+-------+---------+-----------+---------+-----------+
|     cursos|   NULL|       []|       NULL|TEMPORARY|       true|
|estudiantes|   NULL|       []|       NULL|TEMPORARY|       true|
| matriculas|   NULL|       []|       NULL|TEMPORARY|       true|
+-----------+-------+---------+-----------+---------+-----------+


RESUMEN — Caso de Estudio EduTrack Academy
✅ CORRECTO — Vistas base registradas (3)
✅ CORRECTO — SQL y API producen el mismo resultado
✅ CORRECTO — IN — matrículas Premium
✅ CORRECTO — EXISTS — cursos con completados
✅ CORRECTO — FROM — ranking de gasto
❌ REVISAR — CTE simple — sin completados
✅ CORRECTO — CTE encadenada — por plan
✅ CORRECTO — Vista derivada eliminada


res42_0: Boolean = true
checks: Seq[(String, Boolean)] = List(
  ("Vistas base registradas (3)", true),
  ("SQL y API producen el mismo resultado", true),
  ("IN — matrículas Premium", true),
  ("EXISTS — cursos con completados", true),
  ("FROM — ranking de gasto", true),
  ("CTE simple — sin completados", false),
  ("CTE encadenada — por plan", true),
  ("Vista derivada eliminada", true)
)

---

## 🛑 Cierre

Descomenta para detener Spark al terminar.


In [42]:
// spark.stop()